In [1]:
import pandas as pd
import numpy as np

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
file_path = "/content/drive/MyDrive/Lion-Biologging-SoS'26/lion_data_with_features.csv"
df = pd.read_csv(file_path)
df.head()

,longitude,latitude,lion_id,local_time,x,y,z,time_delta,movement_valid,dx,...,dz,distance,bearing,turn,bearing_sin,bearing_cos,turn_sin,turn_cos,hour_sin,hour_cos
0,22.743923,-21.286270,1001,2009-12-07 22:00:00,0.859325,0.360238,-0.363028,NaN,False,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.500000,0.866025
1,22.743892,-21.286279,1001,2009-12-07 23:00:00,0.859325,0.360237,-0.363028,0 days 01:00:00,True,1.423170e-07,...,-1.463634e-07,5.280452e-07,-1.286473,NaN,-0.959852,0.280508,NaN,NaN,-0.258819,0.965926
2,22.743804,-21.286048,1001,2009-12-08 00:00:00,0.859327,0.360236,-0.363024,0 days 01:00:00,True,1.903094e-06,...,3.756663e-06,4.278173e-06,-0.377214,0.909260,-0.368332,0.929694,0.789049,0.614330,0.000000,1.000000
3,22.744277,-21.287271,1001,2009-12-08 01:00:00,0.859317,0.360241,-0.363044,0 days 01:00:00,True,-1.012046e-05,...,-1.988911e-05,2.268909e-05,2.756839,3.134053,0.375331,-0.926891,0.007540,-0.999972,0.258819,0.965926
4,22.744216,-21.287307,1001,2009-12-08 02:00:00,0.859317,0.360240,-0.363045,0 days 01:00:00,True,1.731600e-07,...,-5.854495e-07,1.174253e-06,-1.399850,2.126497,-0.985424,0.170115,0.849531,-0.527538,0.500000,0.866025


In [4]:
#defining constants
window_size = 24 #because lion have circadian rythm (24hr cycle)
stride = 1 #window moves forward 1hour at a time

In [13]:
#initializing container X for storing 24hr windows
X_sequences = []

In [15]:
for lion_id, lion_sub_df in df.groupby('lion_id'):
  total_rows = len(lion_sub_df)
  stopping_point = total_rows - window_size + 1

  for i in range(0,stopping_point,stride):
    window = lion_sub_df.iloc[i:i+window_size]

    if not window['movement_valid'].all():
      continue

    filtered_window = window[['distance', 'dz','bearing_sin', 'bearing_cos','turn_sin', 'turn_cos','hour_sin', 'hour_cos']]
    X_sequences.append(filtered_window.values)


In [17]:
#converting the 2d arrays stored in X_sequences into a single 3D tensor matrix
X_3D = np.array(X_sequences)

In [19]:
#saving our 3D tensor as a binary file to use in next week's notebook
save_path = "/content/drive/MyDrive/Lion-Biologging-SoS'26/lion_sequences_X_3d.npy"
np.save(save_path, X_3D)